byte_tracker.py

In [1]:
from trackers.byte_tracker import BYTETracker

import cv2
import torch
import torchvision
import torchvision.transforms as T
import numpy as np
from PIL import Image
from collections import defaultdict
import time
from torchvision.models.detection import ssdlite320_mobilenet_v3_large, SSDLite320_MobileNet_V3_Large_Weights
from types import SimpleNamespace

In [2]:

import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [3]:
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

# Define the transform (same as during training)
transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225])
])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and modify the detection model
model = ssdlite320_mobilenet_v3_large(
    weights=None,
    num_classes=2,
    weights_backbone=torchvision.models.MobileNet_V3_Large_Weights.IMAGENET1K_V1
)
model_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Model.pth"
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()


SSD(
  (backbone): SSDLiteFeatureExtractorMobileNet(
    (features): Sequential(
      (0): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
          (2): Hardswish()
        )
        (1): InvertedResidual(
          (block): Sequential(
            (0): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
              (2): ReLU(inplace=True)
            )
            (1): Conv2dNormActivation(
              (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
              (1): BatchNorm2d(16, eps=0.001, momentum=0.03, affine=True, track_running_stats=True)
            )
          )
        )
        (2): Invert

Helper Functions

In [4]:
def xyxy_to_xywh(xyxy):
    """Convert bounding box from [x1, y1, x2, y2] to [cx, cy, w, h]."""
    x1, y1, x2, y2 = xyxy[:4]
    cx = (x1 + x2) / 2.0
    cy = (y1 + y2) / 2.0
    w = x2 - x1
    h = y2 - y1
    return np.array([cx, cy, w, h], dtype=xyxy.dtype)

def predict(image_input, model, device, threshold=0.2, nms_threshold=0.2, max_detections=1):
    """
    Predict detections on an image using the SSDLite model.
    
    Args:
        image_input (str or np.ndarray): File path or OpenCV BGR image.
        model: The detection model.
        device: Computation device.
        threshold (float): Confidence threshold.
        nms_threshold (float): (Currently unused) IoU threshold for NMS.
        max_detections (int): Maximum number of top scoring detections to return.
        
    Returns:
        orig_img (np.ndarray): Original image (BGR).
        detections (np.ndarray): Array of detections in [x1, y1, x2, y2, score] format.
        inference_time (float): Inference time in milliseconds.
    """
    from PIL import Image
    import torchvision.transforms as T

    # Define transform (should match training)
    transform = T.Compose([
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406],
                    std=[0.229, 0.224, 0.225])
    ])

    # Convert input to PIL image and keep a copy in BGR for display
    if isinstance(image_input, np.ndarray):
        pil_img = Image.fromarray(cv2.cvtColor(image_input, cv2.COLOR_BGR2RGB))
        orig_img = image_input.copy()
    elif isinstance(image_input, str):
        pil_img = Image.open(image_input).convert("RGB")
        orig_img = cv2.cvtColor(np.array(pil_img), cv2.COLOR_RGB2BGR)
    else:
        raise ValueError("Unsupported image input type.")

    img_tensor = transform(pil_img).to(device)
    img_tensor = img_tensor.unsqueeze(0)

    start = cv2.getTickCount()
    with torch.no_grad():
        outputs = model(img_tensor)
    end = cv2.getTickCount()
    inference_time = (end - start) / cv2.getTickFrequency() * 1000.0
    print(f"Inference time: {inference_time:.1f} ms")

    output = outputs[0]
    boxes = output['boxes'].cpu().numpy()   # shape: (N, 4)
    scores = output['scores'].cpu().numpy()   # shape: (N,)

    # Filter detections below threshold
    keep = scores >= threshold
    boxes = boxes[keep]
    scores = scores[keep]

    if len(boxes) > 0:
        # Get indices that would sort scores in descending order
        sorted_indices = np.argsort(scores)[::-1]
        # Select the top `max_detections` indices
        top_indices = sorted_indices[:max_detections]
        boxes = boxes[top_indices]
        scores = scores[top_indices]
    else:
        detections = np.empty((0, 5), dtype=np.float32)
        return orig_img, detections, inference_time

    detections = np.hstack((boxes, scores.reshape(-1, 1)))
    return orig_img, detections, inference_time



In [5]:
def create_dummy_results(dets):
    """
    Given dets (an array of shape (N, 5) with [x1, y1, x2, y2, score]),
    return a dummy results object with attributes:
      - xywh: bounding boxes in [cx, cy, w, h] format
      - conf: confidence scores
      - cls: class labels (assumed to be 0 for all detections)
    """
    if len(dets) == 0:
        return SimpleNamespace(
            xywh=np.empty((0, 4), dtype=np.float32),
            conf=np.empty((0,), dtype=np.float32),
            cls=np.empty((0,), dtype=np.int32)
        )
    boxes_xywh = np.array([xyxy_to_xywh(det[:4]) for det in dets])
    conf = dets[:, 4]
    cls = np.zeros_like(conf, dtype=np.int32)
    return SimpleNamespace(xywh=boxes_xywh, conf=conf, cls=cls)

 Tracker Setup
 Here's a breakdown of each parameter in your DummyArgs class and how it affects the tracker:

track_thresh (0.2):
This is the minimum detection confidence required for a detection to be considered for tracking. Detections with a score below this value will be ignored in the tracking process. In your case, only detections with confidence ≥ 0.2 are used for initializing or updating tracks.

track_buffer (40):
This parameter defines the maximum number of frames a track can remain “lost” (i.e. not updated with a new detection) before it is removed from the active track list. A higher value allows tracks to be maintained longer even if an object is temporarily occluded or missed.

match_thresh (0.8):
This is the threshold for matching a new detection to an existing track—often based on Intersection over Union (IoU) or a similar cost metric. If the cost between a track's predicted position and a detection is below this threshold (or equivalently, if the similarity is above a certain level), the detection is considered a match for that track.

mot20 (False):
This flag indicates whether to use MOT20-specific settings. MOT20 is a benchmark for multi-object tracking with specific challenges. When set to True, the tracker might adjust certain thresholds or processing steps to suit MOT20 scenarios.

track_low_thresh (0.1):
This is a lower confidence threshold used during a secondary association step. Detections with confidence above 0.1 (but below the high threshold) might be used to update tracks that weren't matched with high-confidence detections, giving a chance to recover missed objects.

track_high_thresh (0.2):
This is the high confidence threshold used in the primary association step. Detections with confidence above 0.2 are considered high-confidence and are prioritized for matching with existing tracks.

new_track_thresh (0.2):
This threshold determines whether a detection is confident enough to initialize a new track. If a detection's score is below this value, even if it doesn't match any existing track, it might not be used to start a new track.

fuse_score (False):
When enabled (set to True), this option instructs the tracker to combine (or “fuse”) the detection score with the matching cost during data association. Fusing the score can sometimes improve association by giving additional weight to high-confidence detections when matching them to existing tracks.


In [6]:
class DummyArgs:
    track_thresh = 0.2        # Minimum detection confidence for tracking
    track_buffer = 100         # Maximum frames to keep a lost track
    match_thresh = 0.9        # Threshold for matching (e.g., IoU)
    mot20 = False             # MOT20-specific settings flag
    track_low_thresh = 0.1    # Low threshold for detections (second association)
    track_high_thresh = 0.2   # High threshold for detections (first association)
    new_track_thresh = 0.0   # Threshold to initialize a new track
    fuse_score = False        # Whether to fuse detection score

args = DummyArgs()
# Import BYTETracker from your trackers package
from trackers.byte_tracker import BYTETracker
tracker = BYTETracker(args=args, frame_rate=15)

In [7]:
import cv2
import numpy as np
import time
from collections import defaultdict

# Assuming predict, create_dummy_results, and BYTETracker (tracker) are already defined and imported.

# video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\TestFile_video.mp4"
video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Baseline.mp4"
# video_path = r"C:\Machine Learning\Rat Tracking using TensorFlow\Video\Baseline.mp4"
cap = cv2.VideoCapture(video_path)
cv2.namedWindow("Tracking", cv2.WINDOW_NORMAL)

# Dictionary to store track history for drawing tracking lines: track_id -> list of (cx, cy)
track_history = defaultdict(list)
start_time_global = time.time()

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    orig_img, detections, inf_time = predict(frame, model, device, threshold=0.2, max_detections=1)
    print("Detections shape:", detections.shape)
    print("Detections:", detections)

    # Create dummy results object from detections.
    results = create_dummy_results(detections)
    # Update tracker (BYTETracker expects a results object with attributes: xywh, conf, and cls)
    tracks = tracker.update(results, img=None)
    print("Tracked outputs:")
    print(tracks)
    
    # Optionally, print track IDs and scores to console.
    if tracks.size > 0:
        for track in tracks:
            print("Track ID:", int(track[4]), "Score:", track[5])
    
    # Annotate the image with detection boxes using dummy results.
    def plot_detections(orig_img, dummy_results):
        annotated_img = orig_img.copy()
        for box, score in zip(dummy_results.xywh, dummy_results.conf):
            cx, cy, w, h = box
            x1 = cx - w / 2
            y1 = cy - h / 2
            x2 = cx + w / 2
            y2 = cy + h / 2
            cv2.rectangle(annotated_img, (int(x1), int(y1)), (int(x2), int(y2)), (255, 0, 0), 2)
        return annotated_img

    annotated_frame = plot_detections(orig_img, results)
    
    # Overlay track lines and display track IDs and scores.
    if tracks.size > 0:
        for track in tracks:
            x1, y1, x2, y2, tid, score, cls, idx = track
            cx = (x1 + x2) / 2
            cy = (y1 + y2) / 2
            tid = int(tid)
            track_history[tid].append((cx, cy))
            if len(track_history[tid]) > 30:
                track_history[tid].pop(0)
            pts = np.array(track_history[tid], dtype=np.int32).reshape(-1, 1, 2)
            cv2.polylines(annotated_frame, [pts], isClosed=False, color=(230, 230, 230), thickness=2)
            label = f"ID:{tid}, Conf: {score:.2f}"
            cv2.putText(annotated_frame, label, (int(x1), int(y1)-15),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)
    
    # Display inference time on the top-left corner.
    cv2.putText(annotated_frame, f"Inference: {inf_time:.1f} ms", (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 255), 2)
    
    cv2.imshow("Tracking", annotated_frame)
    if cv2.waitKey(1) & 0xFF == ord('q'):
        print("Exiting display loop.")
        break

cap.release()
cv2.destroyAllWindows()
print("Video processing complete.")


Inference time: 747.1 ms
Detections shape: (1, 5)
Detections: [[430.42117   125.11832   576.2175    373.87344     0.9999361]]
Tracked outputs:
[[430.42114   125.11832   576.2175    373.87344     1.          0.9999361
    0.          0.       ]]
Track ID: 1 Score: 0.9999361
Inference time: 22.9 ms
Detections shape: (1, 5)
Detections: [[433.19302   125.92499   575.1064    375.72467     0.9999014]]
Tracked outputs:
[[430.92017   125.81832   577.15967   375.4799      1.          0.9999014
    0.          0.       ]]
Track ID: 1 Score: 0.9999014
Inference time: 22.5 ms
Detections shape: (1, 5)
Detections: [[431.5706    124.844025  573.9587    374.6537      0.9999169]]
Tracked outputs:
[[429.97443   125.09691   576.1962    374.92157     1.          0.9999169
    0.          0.       ]]
Track ID: 1 Score: 0.9999169
Inference time: 24.3 ms
Detections shape: (1, 5)
Detections: [[432.17935    122.78602    574.56995    375.07153      0.99990416]]
Tracked outputs:
[[429.66095    123.28737    576.8